# LensForge LSST Mock Pipeline

This notebook documents the runnable LSST-style packaging workflow in LensForge. It takes a mock survey subset, packages it into DeepLense-ready folders, and verifies that the packaged output can be consumed by the Test V trainer.

In [ ]:
from pathlib import Path
import json

repo_root = Path.cwd().resolve()
task_data_root = repo_root / 'data' / 'lens-finding-test'
output_root = repo_root / 'tmp' / 'lsst_mock_pipeline'
report_path = repo_root / 'reports' / 'lsst_mock_pipeline_run.json'
smoke_report_path = repo_root / 'reports' / 'lsst_pipeline_smoke.json'

repo_root, task_data_root, output_root

## Run the pipeline

Use the CLI entry point below after placing the Test V dataset under `data/lens-finding-test`.

In [ ]:
print(
    "python3 run_lsst_mock_pipeline.py "
    "--data-root data/lens-finding-test "
    "--output-root tmp/lsst_mock_pipeline "
    "--max-per-folder 16 "
    "--report-path reports/lsst_mock_pipeline_run.json"
)

In [ ]:
pipeline_report = json.loads(report_path.read_text())
pipeline_report

## Packaged artifact layout

In [ ]:
packaged_root = output_root / 'deeplense_dataset'
sorted(path.name for path in packaged_root.iterdir())

## Downstream smoke check

A one-epoch smoke run verifies that the packaged output can go straight into the Test V training pipeline.

In [ ]:
print(
    "python3 train.py "
    "--data-root tmp/lsst_mock_pipeline/deeplense_dataset "
    "--epochs 1 --batch-size 16 --balance-strategy both "
    "--model-path models/lsst_pipeline_smoke.pt "
    "--report-path reports/lsst_pipeline_smoke.json"
)

In [ ]:
smoke_report = json.loads(smoke_report_path.read_text())
{
    'test_roc_auc': smoke_report['test']['roc_auc'],
    'test_pr_auc': smoke_report['test']['pr_auc'],
    'test_recall': smoke_report['test']['recall'],
}